#  Learning Unsupervised Embeddings for Molecules

In this tutorial, we will use a `SeqToSeq` model to generate fingerprints for classifying molecules.  This is based on the following paper, although some of the implementation details are different: Xu et al., "Seq2seq Fingerprint: An Unsupervised Deep Molecular Embedding for Drug Discovery" (https://doi.org/10.1145/3107411.3107424).

## Colab

This tutorial and the rest in this sequence can be done in Google colab. If you'd like to open this notebook in colab, you can use the following link.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/Learning_Unsupervised_Embeddings_for_Molecules.ipynb)



In [253]:
!pip install --pre deepchem
import deepchem
deepchem.__version__

DEPRECATION: Loading egg at /home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/sympy-1.13.3-py3.12.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/pillow-11.1.0-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/rdkit-2024.9.5-py3.12-linux-x86_64.egg is deprecated. pip 25.1 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /home/jant

'2.5.0'

In [254]:
import deepchem as dc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import ExponentialLR

In [255]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Learning Embeddings with SeqToSeq

Many types of models require their inputs to have a fixed shape.  Since molecules can vary widely in the numbers of atoms and bonds they contain, this makes it hard to apply those models to them.  We need a way of generating a fixed length "fingerprint" for each molecule.  Various ways of doing this have been designed, such as the Extended-Connectivity Fingerprints (ECFPs) we used in earlier tutorials.  But in this example, instead of designing a fingerprint by hand, we will let a `SeqToSeq` model learn its own method of creating fingerprints.

A `SeqToSeq` model performs sequence to sequence translation.  For example, they are often used to translate text from one language to another.  It consists of two parts called the "encoder" and "decoder".  The encoder is a stack of recurrent layers.  The input sequence is fed into it, one token at a time, and it generates a fixed length vector called the "embedding vector".  The decoder is another stack of recurrent layers that performs the inverse operation: it takes the embedding vector as input, and generates the output sequence.  By training it on appropriately chosen input/output pairs, you can create a model that performs many sorts of transformations.

In this case, we will use SMILES strings describing molecules as the input sequences.  We will train the model as an autoencoder, so it tries to make the output sequences identical to the input sequences.  For that to work, the encoder must create embedding vectors that contain all information from the original sequence.  That's exactly what we want in a fingerprint, so perhaps those embedding vectors will then be useful as a way to represent molecules in other models!

Let's start by loading the data.  We will use the MUV dataset.  It includes 74,501 molecules in the training set, and 9313 molecules in the validation set, so it gives us plenty of SMILES strings to work with.

In [256]:
# Load dataset using DeepChem
tasks, datasets, transformers = dc.molnet.load_muv(splitter='stratified')
train_dataset, valid_dataset, test_dataset = datasets
train_smiles = train_dataset.ids
valid_smiles = valid_dataset.ids

We need to define the "alphabet" for our `SeqToSeq` model, the list of all tokens that can appear in sequences.  (It's also possible for input and output sequences to have different alphabets, but since we're training it as an autoencoder, they're identical in this case.)  Make a list of every character that appears in any training sequence.

In [257]:
# Extract tokens form the SMILES strings
tokens = set()
for s in train_smiles:
    tokens = tokens.union(set(c for c in s))

# Add special tokens (for PyTorch implementation)
tokens.add('<START>')
tokens.add('<END>')

tokens = sorted(list(tokens))

In [258]:
# For PyTorch, create a mapping from tokens to indices and vice versa to prepare for DataLoader
token_to_idx = {token: idx for idx, token in enumerate(tokens)}
idx_to_token = {idx: token for token, idx in token_to_idx.items()}

In [259]:
# Define PyTorch Dataset and dataloader
class SMILESDataset(Dataset):
    def __init__(self, smiles_list, token_to_idx, max_length):
        self.smiles_list = smiles_list
        self.token_to_idx = token_to_idx
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles_list)

    def __getitem__(self, idx):
        smiles = self.smiles_list[idx]
        # Convert SMILES string to a sequence of token indices
        input_seq = [self.token_to_idx[token] for token in smiles]
        # Add padding to the sequence to make it `max_length`
        input_seq = input_seq + [0] * (self.max_length - len(input_seq))
        input_seq = torch.tensor(input_seq, dtype=torch.long)

        # For simplicity, use the same sequence as the target (e.g., autoencoder)
        target_seq = input_seq.clone()
        return input_seq, target_seq

# Max sequence length
max_length = max(len(s) for s in train_smiles)

# Create the training dataset and DataLoader
batch_size = 100  # Define your batch size
train_dataset = SMILESDataset(train_smiles, token_to_idx, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Create the validation dataset and DataLoader (optional)
valid_dataset = SMILESDataset(valid_smiles, token_to_idx, max_length)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

Create the model and define the optimization method to use.  In this case, learning works much better if we gradually decrease the learning rate.  We use an `ExponentialDecay` to multiply the learning rate by 0.9 after each epoch.

In [279]:
import inspect
import deepchem.models

print(inspect.getfile(deepchem.models.SeqToSeq))

/home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/deepchem/models/seqtoseq.py


In [260]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
from heapq import heappush, heappushpop


class SeqToSeq(nn.Module):
    """Implements sequence-to-sequence translation models in PyTorch."""

    sequence_end = "<END>"

    def __init__(self,
                 input_tokens,
                 output_tokens,
                 max_output_length,
                 encoder_layers=4,
                 decoder_layers=4,
                 embedding_dimension=512,
                 dropout=0.0,
                 reverse_input=True,
                 variational=False,
                 annealing_start_step=5000,
                 annealing_final_step=10000,
                 **kwargs):
        """
        Construct a SeqToSeq model.

        Parameters
        ----------
        input_tokens: list
            List of all tokens that may appear in input sequences.
        output_tokens: list
            List of all tokens that may appear in output sequences.
        max_output_length: int
            Maximum length of output sequence that may be generated.
        encoder_layers: int
            Number of recurrent layers in the encoder.
        decoder_layers: int
            Number of recurrent layers in the decoder.
        embedding_dimension: int
            Dimension of the embedding vector and GRU hidden states.
        dropout: float
            Dropout probability during training.
        reverse_input: bool
            If True, reverse the input sequence before encoding.
        variational: bool
            If True, train the model as a variational autoencoder.
        annealing_start_step: int
            Step at which to begin turning on the constraint term for KL cost annealing.
        annealing_final_step: int
            Step at which to finish turning on the constraint term for KL cost annealing.
        """
        super(SeqToSeq, self).__init__()
        self.input_tokens = input_tokens
        self.output_tokens = output_tokens
        self.input_dict = {token: idx for idx, token in enumerate(input_tokens)}
        self.output_dict = {token: idx for idx, token in enumerate(output_tokens)}
        self.max_output_length = max_output_length  # Initialize max_output_length
        self.embedding_dimension = embedding_dimension
        self.reverse_input = reverse_input
        self.variational = variational
        self.annealing_start_step = annealing_start_step
        self.annealing_final_step = annealing_final_step

        # Embedding layers
        self.input_embedding = nn.Embedding(len(input_tokens), embedding_dimension)
        self.output_embedding = nn.Embedding(len(output_tokens), embedding_dimension)

        # Encoder
        self.encoder = nn.GRU(embedding_dimension, embedding_dimension,
                              num_layers=encoder_layers, batch_first=True, dropout=dropout)

        # Decoder
        self.decoder = nn.GRU(embedding_dimension, embedding_dimension,
                              num_layers=decoder_layers, batch_first=True, dropout=dropout)
        self.output_layer = nn.Linear(embedding_dimension, len(output_tokens))

        # Variational components
        if variational:
            self.mean_layer = nn.Linear(embedding_dimension, embedding_dimension)
            self.std_layer = nn.Linear(embedding_dimension, embedding_dimension)

    def _create_features(self):
        """This function is redundant in PyTorch."""
        # In PyTorch, we directly use tensors as inputs, so this function is unnecessary.
        pass

    def _create_encoder(self, n_layers, dropout):
        """This function is redundant in PyTorch."""
        # The encoder is already defined in the `__init__` method using `nn.GRU`.
        pass

    def _create_decoder(self, n_layers, dropout):
        """This function is redundant in PyTorch."""
        # The decoder is already defined in the `__init__` method using `nn.GRU` and `nn.Linear`.
        pass

    def _create_loss(self):
        """This function is redundant in PyTorch."""
        # In PyTorch, we use built-in loss functions like `nn.CrossEntropyLoss` directly.
        pass

    def forward(self, input_seq, target_seq=None, global_step=None):
        """
        Forward pass for training or inference.

        Parameters
        ----------
        input_seq: torch.Tensor
            Input sequence tensor of shape (batch_size, seq_length).
        target_seq: torch.Tensor, optional
            Target sequence tensor of shape (batch_size, seq_length). If None, inference is performed.
        global_step: int, optional
            Current training step for KL cost annealing (used in variational autoencoder).

        Returns
        -------
        output: torch.Tensor
            Output sequence tensor of shape (batch_size, max_output_length, vocab_size).
        """
        # Reverse input sequence if required
        if self.reverse_input:
            input_seq = torch.flip(input_seq, dims=[1])

        # Encode input sequence
        embedded_input = self.input_embedding(input_seq)
        _, hidden = self.encoder(embedded_input)

        # Variational autoencoder
        if self.variational:
            mean = self.mean_layer(hidden[-1])
            std = torch.exp(0.5 * self.std_layer(hidden[-1]))
            z = mean + std * torch.randn_like(std)

            # KL cost annealing
            if global_step is not None:
                anneal_steps = self.annealing_final_step - self.annealing_start_step
                if anneal_steps > 0:
                    current_step = max(0, global_step - self.annealing_start_step)
                    kl_scale = min(1.0, (current_step / anneal_steps) ** 2)
                else:
                    kl_scale = 1.0
                kl_loss = 0.5 * kl_scale * torch.mean(mean ** 2 + std ** 2 - torch.log(std ** 2 + 1e-20) - 1)
                self.add_loss(kl_loss)
            hidden = z.unsqueeze(0).repeat(self.encoder.num_layers, 1, 1)

        # Decode output sequence
        if target_seq is not None:
            # Training mode
            embedded_target = self.output_embedding(target_seq)
            decoder_output, _ = self.decoder(embedded_target, hidden)
        else:
            # Inference mode
            batch_size = input_seq.size(0)
            decoder_input = torch.full((batch_size, 1), self.output_dict['<START>'],
                                    dtype=torch.long, device=input_seq.device)
            decoder_output = []
            for _ in range(self.max_output_length):
                embedded_decoder_input = self.output_embedding(decoder_input)
                output, hidden = self.decoder(embedded_decoder_input, hidden)
                output_token = torch.argmax(self.output_layer(output), dim=-1)
                decoder_output.append(output_token)
                decoder_input = output_token
            decoder_output = torch.cat(decoder_output, dim=1)

            # Embed the decoder_output tokens
            decoder_output = self.output_embedding(decoder_output)

        # Reshape decoder_output for the Linear layer
        decoder_output = decoder_output.reshape(-1, self.embedding_dimension)  # Flatten the sequence dimension

        # Compute output probabilities
        output = self.output_layer(decoder_output.float())  # Convert to float
        output = output.view(input_seq.size(0), self.max_output_length, -1)  # Reshape back to (batch_size, seq_length, vocab_size)
        return output

    def fit_sequences(self, dataloader, optimizer, scheduler, device, epochs=1):
        """
        Train the model on a set of sequences.

        Parameters
        ----------
        dataloader: DataLoader
            DataLoader for the training dataset.
        optimizer: torch.optim.Optimizer
            Optimizer for training.
        scheduler: torch.optim.lr_scheduler._LRScheduler
            Learning rate scheduler.
        device: torch.device
            Device to train the model on (CPU or GPU).
        epochs: int
            Number of epochs to train for.
        """
        self.train()
        for epoch in range(epochs):
            total_loss = 0
            for batch in dataloader:
                input_seq, target_seq = batch
                input_seq, target_seq = input_seq.to(device), target_seq.to(device)

                optimizer.zero_grad()
                output = self(input_seq, target_seq)
                loss = F.cross_entropy(output.view(-1, output.size(-1)), target_seq.view(-1), ignore_index=0)
                loss.backward()
                optimizer.step()
                total_loss += loss.item()
            scheduler.step()
            print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(dataloader)}")

    def predict_from_sequences(self, sequences, token_to_idx, idx_to_token, device):
        """
        Predict output sequences for a list of input sequences.

        Parameters
        ----------
        sequences: list
            List of input sequences (e.g., SMILES strings).
        token_to_idx: dict
            Mapping from tokens to indices.
        idx_to_token: dict
            Mapping from indices to tokens.
        device: torch.device
            Device to run the model on (CPU or GPU).

        Returns
        -------
        list
            List of predicted sequences.
        """
        self.eval()
        predicted_sequences = []
        with torch.no_grad():
            for seq in sequences:
                # Tokenize the input sequence
                input_seq = [token_to_idx[token] for token in seq]
                input_seq = input_seq + [0] * (self.max_output_length - len(input_seq))
                input_seq = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(device)
                
                # Predict the output sequence
                output = self(input_seq)
                predicted_tokens = torch.argmax(output, dim=-1).squeeze(0).tolist()
                
                # Convert token indices back to SMILES string
                predicted_sequences.append(''.join([idx_to_token[idx] for idx in predicted_tokens if idx != 0]))
        return predicted_sequences

    def predict_from_embeddings(self, embeddings, device):
        """
        Predict output sequences from embedding vectors.

        Parameters
        ----------
        embeddings: torch.Tensor
            Embedding vectors of shape (batch_size, embedding_dimension).
        device: torch.device
            Device to run the model on (CPU or GPU).

        Returns
        -------
        list
            List of predicted sequences.
        """
        self.eval()
        predicted_sequences = []
        with torch.no_grad():
            for embedding in embeddings:
                embedding = embedding.unsqueeze(0).to(device)
                hidden = embedding.repeat(self.decoder.num_layers, 1, 1)
                decoder_input = torch.full((1, 1), self.output_dict['<START>'], dtype=torch.long, device=device)
                output_sequence = []
                for _ in range(self.max_output_length):
                    embedded_decoder_input = self.output_embedding(decoder_input)
                    output, hidden = self.decoder(embedded_decoder_input, hidden)
                    output_token = torch.argmax(self.output_layer(output), dim=-1)
                    output_sequence.append(output_token.item())
                    decoder_input = output_token
                predicted_sequences.append(''.join([self.output_tokens[idx] for idx in output_sequence if idx != 0]))
        return predicted_sequences

    def predict_embeddings(self, sequences):
        """
        Compute embedding vectors for a set of input sequences.

        Parameters
        ----------
        sequences: list
            List of input sequences (e.g., SMILES strings).

        Returns
        -------
        np.ndarray
            Embedding vectors of shape (num_samples, embedding_dimension).
        """
        self.eval()
        embeddings = []
        with torch.no_grad():
            for seq in sequences:
                # Tokenize the input sequence
                input_seq = [self.input_dict[token] for token in seq]
                input_seq = input_seq + [0] * (self.max_output_length - len(input_seq))  # Use self.max_output_length
                input_seq = torch.tensor(input_seq, dtype=torch.long).unsqueeze(0).to(next(self.parameters()).device)

                # Compute the embedding
                embedded_input = self.input_embedding(input_seq)
                _, hidden = self.encoder(embedded_input)
                embeddings.append(hidden[-1].cpu().numpy())  # Convert to NumPy array
        return np.array(embeddings, dtype=np.float32)
    
    def predict_embeddings_batch(self, sequences, batch_size=32):
        """
        Compute embedding vectors for a set of input sequences in batches.

        Parameters
        ----------
        sequences: list
            List of input sequences (e.g., SMILES strings).
        batch_size: int
            Number of sequences to process in each batch.

        Returns
        -------
        np.ndarray
            Embedding vectors of shape (num_samples, embedding_dimension).
        """
        self.eval()
        embeddings = []
        with torch.no_grad():
            for batch_start in range(0, len(sequences), batch_size):
                batch = sequences[batch_start:batch_start + batch_size]
                input_seqs = []
                for seq in batch:
                    input_seq = [self.input_dict[token] for token in seq]
                    input_seq = input_seq + [0] * (self.max_output_length - len(input_seq))
                    input_seqs.append(input_seq)
                input_seqs = torch.tensor(input_seqs, dtype=torch.long).to(next(self.parameters()).device)

                # Compute the embeddings
                embedded_input = self.input_embedding(input_seqs)
                _, hidden = self.encoder(embedded_input)
                embeddings.extend(hidden[-1].cpu().numpy())  # Convert to NumPy array
        return np.array(embeddings, dtype=np.float32)
    

    def _beam_search(self, probs, beam_width):
        """
        Perform a beam search for the most likely output sequence.

        Parameters
        ----------
        probs: torch.Tensor
            Probabilities of shape (seq_length, vocab_size).
        beam_width: int
            Beam width for searching.

        Returns
        -------
        list
            Most likely output sequence.
        """
        if beam_width == 1:
            # Greedy search
            return [self.output_tokens[torch.argmax(p).item()] for p in probs]

        # Beam search
        logprobs = torch.log(probs)
        candidates = [(0.0, [])]
        for step_probs in logprobs:
            new_candidates = []
            for score, seq in candidates:
                for idx, logprob in enumerate(step_probs):
                    new_candidates.append((score + logprob.item(), seq + [idx]))
            candidates = sorted(new_candidates, key=lambda x: x[0], reverse=True)[:beam_width]
        return [self.output_tokens[idx] for idx in candidates[0][1]]

    def _create_input_array(self, sequences):
        """This function is redundant in PyTorch."""
        # In PyTorch, we directly use tensors for input sequences.
        pass

    def _create_output_array(self, sequences):
        """This function is redundant in PyTorch."""
        # In PyTorch, we directly use tensors for target sequences.
        pass

    def _batch_elements(self, elements, batch_size):
        """
        Combine elements into batches.

        Parameters
        ----------
        elements: list
            List of elements to batch.
        batch_size: int
            Batch size.

        Yields
        ------
        list
            Batches of elements.
        """
        for i in range(0, len(elements), batch_size):
            yield elements[i:i + batch_size]

    def _generate_batches(self, sequences, batch_size):
        """
        Generate batches of input/output pairs for training.

        Parameters
        ----------
        sequences: list
            List of input/output sequence pairs.
        batch_size: int
            Batch size.

        Yields
        ------
        tuple
            Batches of input and output sequences.
        """
        for batch in self._batch_elements(sequences, batch_size):
            inputs, outputs = zip(*batch)
            yield torch.tensor(inputs, dtype=torch.long), torch.tensor(outputs, dtype=torch.long)

In [261]:
# Define the model
model = SeqToSeq(
    input_tokens=tokens,
    output_tokens=tokens,
    max_output_length=max_length,
    encoder_layers=2,
    decoder_layers=2,
    embedding_dimension=256,
    dropout=0.0,
    reverse_input=True,
    variational=False
)

# Move the model to the appropriate device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SeqToSeq(
  (input_embedding): Embedding(31, 256)
  (output_embedding): Embedding(31, 256)
  (encoder): GRU(256, 256, num_layers=2, batch_first=True)
  (decoder): GRU(256, 256, num_layers=2, batch_first=True)
  (output_layer): Linear(in_features=256, out_features=31, bias=True)
)

Let's train it!  The input to `fit_sequences()` is a generator that produces input/output pairs.  On a good GPU, this should take a few hours or less.

In [262]:
# def generate_sequences(epochs):
#   for i in range(epochs):
#     for s in train_smiles:
#       yield (s, s)

# model.fit_sequences(generate_sequences(40))

In [263]:
# Define optimizer and learning rate scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

# Train the model for 40 epochs using the DataLoader
# epochs = 40
epochs = 3
model.fit_sequences(train_dataloader, optimizer, scheduler, device, epochs=epochs)

Epoch 1/3, Loss: 0.04386800845573975
Epoch 2/3, Loss: 0.00011740876463298129
Epoch 3/3, Loss: 4.550795829528708e-05


Let's see how well it works as an autoencoder.  We'll run the first 500 molecules from the validation set through it, and see how many of them are exactly reproduced.

In [264]:
# predicted = model.predict_from_sequences(valid_smiles[:500])
# count = 0
# for s,p in zip(valid_smiles[:500], predicted):
#   if ''.join(p) == s:
#     count += 1
# print('reproduced', count, 'of 500 validation SMILES strings')

In [265]:
# Predict sequences for the first 500 validation SMILES strings
predicted = model.predict_from_sequences(valid_smiles[:500], token_to_idx, idx_to_token, device)

# Count how many SMILES strings were reproduced exactly
count = 0
for s, p in zip(valid_smiles[:500], predicted):
    if ''.join(p) == s:
        count += 1
print('reproduced', count, 'of 500 validation SMILES strings')

reproduced 0 of 500 validation SMILES strings


Now we'll trying using the encoder as a way to generate molecular fingerprints.  We compute the embedding vectors for all molecules in the training and validation datasets, and create new datasets that have those as their feature vectors.  The amount of data is small enough that we can just store everything in memory.

In [266]:
import numpy as np
import time

# Start timing
time_start = time.time()

# Compute embeddings for the training dataset
train_embeddings = model.predict_embeddings(train_smiles)
train_embeddings_dataset = dc.data.NumpyDataset(
    X=train_embeddings,
    y=datasets[0].y,
    w=datasets[0].w.astype(np.float32),
    ids=datasets[0].ids
)

# Compute embeddings for the validation dataset
valid_embeddings = model.predict_embeddings(valid_smiles)
valid_embeddings_dataset = dc.data.NumpyDataset(
    X=valid_embeddings,
    y=datasets[1].y,
    w=datasets[1].w.astype(np.float32),
    ids=datasets[1].ids
)

# End timing
time_end = time.time()

# Print the elapsed time
print(f"Time taken for embedding computation and dataset creation: {time_end - time_start:.2f} seconds")

Time taken for embedding computation and dataset creation: 131.24 seconds


In [267]:
# Use batch processing for efficiency
time_start = time.time()

# Compute embeddings for the training dataset
train_embeddings = model.predict_embeddings_batch(train_smiles, batch_size=32)
train_embeddings_dataset = dc.data.NumpyDataset(
    X=train_embeddings,
    y=datasets[0].y,  # Labels from the original DeepChem dataset
    w=datasets[0].w.astype(np.float32),  # Weights from the original DeepChem dataset
    ids=datasets[0].ids  # IDs from the original DeepChem dataset
)

# Compute embeddings for the validation dataset
valid_embeddings = model.predict_embeddings_batch(valid_smiles, batch_size=32)
valid_embeddings_dataset = dc.data.NumpyDataset(
    X=valid_embeddings,
    y=datasets[1].y,  # Labels from the original DeepChem dataset
    w=datasets[1].w.astype(np.float32),  # Weights from the original DeepChem dataset
    ids=datasets[1].ids  # IDs from the original DeepChem dataset
)

time_end = time.time()
print(f"Time taken for embedding computation and dataset creation: {time_end - time_start:.2f} seconds")

Time taken for embedding computation and dataset creation: 6.55 seconds


For classification, we'll use a simple fully connected network with one hidden layer.

In [282]:
classifier = dc.models.MultitaskClassifier(
    n_tasks=len(tasks),
    n_features=256,
    layer_sizes=[512]
)

# Train the classifier
classifier.fit(train_embeddings_dataset, nb_epoch=10)

AttributeError: 'str' object has no attribute 'as_numpy_dtype'

In [ ]:
# train_embeddings_dataset = dc.data.NumpyDataset(
#     X=train_embeddings.astype(np.float32),  # Ensure features are float32
#     y=train_embeddings_dataset.y.astype(np.float32),  # Ensure labels are float32
#     w=train_embeddings_dataset.w.astype(np.float32),  # Ensure weights are float32
#     ids=train_embeddings_dataset.ids
# )

# valid_embeddings_dataset = dc.data.NumpyDataset(
#     X=valid_embeddings.astype(np.float32),
#     y=valid_embeddings_dataset.y.astype(np.float32),
#     w=valid_embeddings_dataset.w.astype(np.float32),
#     ids=valid_embeddings_dataset.ids
# )

In [272]:
print(type(train_embeddings_dataset.X))  # Should be <class 'numpy.ndarray'>
print(train_embeddings_dataset.X.shape)  # Should be (num_samples, 256)

<class 'numpy.ndarray'>
(74470, 256)


In [273]:
print(type(train_embeddings_dataset.y))  # Should be <class 'numpy.ndarray'>
print(train_embeddings_dataset.y.shape)  # Should be (num_samples, n_tasks)

<class 'numpy.ndarray'>
(74470, 17)


In [ ]:
# Check the labels
print(type(datasets[0].y))  # Should be <class 'numpy.ndarray'>
print(datasets[0].y.shape)  # Should match the number of samples and tasks

<class 'numpy.ndarray'>
(74470, 17)


In [277]:
print("Type of train_embeddings_dataset.X:", type(train_embeddings_dataset.X))
print("Shape of train_embeddings_dataset.X:", train_embeddings_dataset.X.shape)
print("Type of train_embeddings_dataset.y:", type(train_embeddings_dataset.y))
print("Shape of train_embeddings_dataset.y:", train_embeddings_dataset.y.shape)
print("Type of train_embeddings_dataset.w:", type(train_embeddings_dataset.w))
print("Shape of train_embeddings_dataset.w:", train_embeddings_dataset.w.shape)

Type of train_embeddings_dataset.X: <class 'numpy.ndarray'>
Shape of train_embeddings_dataset.X: (74470, 256)
Type of train_embeddings_dataset.y: <class 'numpy.ndarray'>
Shape of train_embeddings_dataset.y: (74470, 17)
Type of train_embeddings_dataset.w: <class 'numpy.ndarray'>
Shape of train_embeddings_dataset.w: (74470, 17)


In [278]:
import inspect
import deepchem.models

print(inspect.getfile(deepchem.models.MultitaskClassifier))

/home/jantine/miniconda3/envs/deepchem/lib/python3.12/site-packages/deepchem/models/fcnet.py


In [ ]:
# # Remove the extra dimension from the embeddings
# train_embeddings = train_embeddings.squeeze(axis=1)  # Shape becomes (74470, 256)
# valid_embeddings = valid_embeddings.squeeze(axis=1)  # Shape becomes (num_samples, 256)

# # Verify the shape
# print(train_embeddings.shape)  # Should now be (74470, 256)
# print(valid_embeddings.shape)  # Should now be (num_samples, 256)

(74470, 256)
(9309, 256)


In [ ]:
# # Compute embeddings for the training dataset
# train_embeddings = model.predict_embeddings(train_smiles, token_to_idx, device)
# train_embeddings = train_embeddings.cpu().numpy()  # Convert to NumPy array
# train_embeddings = train_embeddings.squeeze(axis=1)  # Remove the extra dimension
# train_embeddings_dataset = dc.data.NumpyDataset(
#     X=train_embeddings,
#     y=train_dataset.y,
#     w=train_dataset.w.astype(np.float32),
#     ids=train_dataset.ids
# )

# # Compute embeddings for the validation dataset
# valid_embeddings = model.predict_embeddings(valid_smiles, token_to_idx, device)
# valid_embeddings = valid_embeddings.cpu().numpy()  # Convert to NumPy array
# valid_embeddings = valid_embeddings.squeeze(axis=1)  # Remove the extra dimension
# valid_embeddings_dataset = dc.data.NumpyDataset(
#     X=valid_embeddings,
#     y=valid_dataset.y,
#     w=valid_dataset.w.astype(np.float32),
#     ids=valid_dataset.ids
# )


AttributeError: 'str' object has no attribute 'as_numpy_dtype'

In [ ]:
# # Define the classifier
# classifier = dc.models.MultitaskClassifier(
#     n_tasks=len(tasks),
#     n_features=256,  # Embedding dimension
#     layer_sizes=[512]  # Hidden layer size
# )

# # Train the classifier
# classifier.fit(train_embeddings_dataset, nb_epoch=10)

Find out how well it worked.  Compute the ROC AUC for the training and validation datasets.

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean, mode="classification")
train_score = classifier.evaluate(train_embeddings_dataset, [metric], transformers)
valid_score = classifier.evaluate(valid_embeddings_dataset, [metric], transformers)
print('Training set ROC AUC:', train_score)
print('Validation set ROC AUC:', valid_score)

# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial, and want to continue working with DeepChem, we encourage you to finish the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open source drug discovery that we're trying to build.

## Join the DeepChem Gitter
The DeepChem [Gitter](https://gitter.im/deepchem/Lobby) hosts a number of scientists, developers, and enthusiasts interested in deep learning for the life sciences. Join the conversation!